# 3 · Real components

Polytropic efficiency, the burner with two gases, nozzles, and the loss bill.
Lesson: [03](../docs/lessons/03-real-components.md) ·
Theory: [component models](../docs/component_models.md).

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from propulsion import plotting as P
from propulsion.flight import FlightCondition
from propulsion.units import FT, tsfc_to_lb_per_lbf_h

P.use_style()
np.set_printoptions(precision=4, suppress=True)
FL350 = FlightCondition.at_altitude(35000 * FT, 0.78, geopotential=True)
SLS = FlightCondition.sea_level_static()
from propulsion.component_models import (Components, turbojet, fuel_air_ratio,
                                         compressor_isentropic_efficiency)
from propulsion.gas import AIR, COMBUSTION_GAS, H_PR_JET_A
CRUISE = FlightCondition.at_altitude(11000.0, 0.8, geopotential=True)

## Polytropic vs isentropic efficiency

In [2]:
for pi in (2.0, 10.0, 30.0, 50.0):
    print(f"pi_c = {pi:4.0f}: eta_c = {float(compressor_isentropic_efficiency(pi, 1.4, 0.90)):.4f}"
          "  (e_c = 0.90)")

pi_c =    2: eta_c = 0.8898  (e_c = 0.90)
pi_c =   10: eta_c = 0.8641  (e_c = 0.90)
pi_c =   30: eta_c = 0.8450  (e_c = 0.90)
pi_c =   50: eta_c = 0.8358  (e_c = 0.90)


## The burner datum

Enthalpies are measured from the heating-value reference temperature (298.15 K). The textbook
form (each gas from 0 K) over-counts the heat when the gases differ.

In [3]:
f = fuel_air_ratio(700.0, 1600.0, AIR, COMBUSTION_GAS, H_PR_JET_A, 0.995)
f0 = fuel_air_ratio(700.0, 1600.0, AIR, COMBUSTION_GAS, H_PR_JET_A, 0.995, T_ref=0.0)
print(f"f (datum 298.15 K) = {float(f):.5f};  f (each gas from 0 K) = {float(f0):.5f}  "
      f"(+{100*(float(f0)/float(f)-1):.1f} %)")

f (datum 298.15 K) = 0.02952;  f (each gas from 0 K) = 0.03151  (+6.8 %)


## Removing the idealisations one at a time

In [4]:
m = Components.modern()
steps = [("ideal (Lesson 2)", dict(components=Components(), hot=AIR, include_fuel_mass=False)),
         ("+ hot gas", dict(components=Components(), include_fuel_mass=False)),
         ("+ fuel mass", dict(components=Components()))]
c = Components()
for name, change in [("+ inlet", dict(pi_d_max=m.pi_d_max)), ("+ compressor", dict(e_c=m.e_c)),
                     ("+ burner", dict(pi_b=m.pi_b, eta_b=m.eta_b)), ("+ turbine", dict(e_t=m.e_t)),
                     ("+ shaft", dict(eta_m=m.eta_m)), ("+ nozzle", dict(pi_n=m.pi_n))]:
    c = c.with_(**change)
    steps.append((name, dict(components=c)))
for name, kw in steps:
    p = turbojet(CRUISE, 20.0, 1600.0, **kw)
    print(f"{name:18s} F/m0 = {float(p.specific_thrust):7.1f}   TSFC = {float(p.tsfc)*1e6:6.2f} mg/(N s)")

ideal (Lesson 2)   F/m0 =   920.9   TSFC =  26.10 mg/(N s)
+ hot gas          F/m0 =   988.2   TSFC =  31.56 mg/(N s)
+ fuel mass        F/m0 =  1036.7   TSFC =  31.26 mg/(N s)
+ inlet            F/m0 =  1035.8   TSFC =  31.29 mg/(N s)
+ compressor       F/m0 =   986.0   TSFC =  31.45 mg/(N s)
+ burner           F/m0 =   978.8   TSFC =  31.85 mg/(N s)
+ turbine          F/m0 =   957.5   TSFC =  32.56 mg/(N s)
+ shaft            F/m0 =   953.9   TSFC =  32.68 mg/(N s)
+ nozzle           F/m0 =   951.9   TSFC =  32.75 mg/(N s)


## Convergent vs fully expanded nozzle

In [5]:
for kind in ("ideal", "convergent"):
    p = turbojet(CRUISE, 20.0, 1600.0, m, nozzle=kind)
    print(f"{kind:10s} F/m0 = {float(p.specific_thrust):6.1f}  M9 = {float(p.extras['M9']):.2f}  "
          f"P9/P0 = {float(p.extras['P9'])/CRUISE.P0:.2f}  eta_P = {float(p.eta_propulsive):.3f}")

ideal      F/m0 =  951.9  M9 = 2.17  P9/P0 = 1.00  eta_P = 0.342
convergent F/m0 =  850.6  M9 = 1.00  P9/P0 = 5.55  eta_P = 0.306
